In [1]:
!pip install -q -U "transformers[serving]" accelerate bitsandbytes

In [2]:
import os
from kaggle_secrets import UserSecretsClient
MODEL = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"   # EDIT: exact HF repo id, org/name, safetensors, no colon
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # only for gated models
assert MODEL != "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic", "set MODEL first"

from huggingface_hub import snapshot_download
path = snapshot_download(MODEL, ignore_patterns=["*.gguf", "*.pth", "original/*"])
print("downloaded:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

downloaded: /root/.cache/huggingface/hub/models--xtools-at--DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic/snapshots/fbccdc6adf70a1981e8f40e91a8370e428e4f998


In [3]:
import os, subprocess, time, requests
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
PROMPT_WORDS = 2500   # stress test size. Set near your real prompt (chars / 4 = tokens)

serve = subprocess.Popen(
    ["transformers", "serve", "--port", "8000",
     "--quantization", "bnb-4bit",
     "--model-timeout", "43200"],   # 12h idle before unload, so no reload mid-session
    stdout=open("serve.log", "w"), stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        requests.get("http://localhost:8000/v1/models", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("server did not start:\n" + open("serve.log").read()[-3000:])

def chat(text, max_tokens=8):
    t = time.time()
    r = requests.post("http://localhost:8000/v1/chat/completions",
        json={"model": MODEL, "messages": [{"role": "user", "content": text}], "max_tokens": max_tokens},
        timeout=1800)
    print(f"  {r.status_code} in {time.time() - t:.0f}s")
    if r.status_code != 200:
        raise RuntimeError(r.text[:500] + "\n--- serve.log ---\n" + open("serve.log").read()[-3000:])

print("loading model into GPU...")
chat("hi")
print(f"stress test, ~{PROMPT_WORDS} word prompt...")
chat("word " * PROMPT_WORDS)
print("READY. Model loaded and survived long prompt.")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

loading model into GPU...
  200 in 54s
stress test, ~2500 word prompt...
  200 in 10s
READY. Model loaded and survived long prompt.
memory.used [MiB], memory.total [MiB]
5001 MiB, 15360 MiB
5415 MiB, 15360 MiB


In [4]:
import re, time, subprocess
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=open("cf.log", "w"),
)
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        URL = m.group(0)
        print("BASE URL:", URL + "/v1")
        break

BASE URL: https://menus-honest-eva-culture.trycloudflare.com/v1


In [5]:
!pip install -q fastapi uvicorn sentence-transformers

In [6]:
%%writefile embed_server.py
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

app = FastAPI()
m = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
m.max_seq_length = 256

class Req(BaseModel):
    model: str | None = None
    input: str | list[str]

@app.post("/v1/embeddings")
def emb(r: Req):
    xs = [r.input] if isinstance(r.input, str) else r.input
    v = m.encode(xs, normalize_embeddings=True, batch_size=16)
    return {"object": "list", "model": "bge-small-en-v1.5",
            "data": [{"object": "embedding", "index": i, "embedding": e.tolist()} for i, e in enumerate(v)],
            "usage": {"prompt_tokens": 0, "total_tokens": 0}}

Writing embed_server.py


In [7]:
subprocess.Popen(["uvicorn", "embed_server:app", "--port", "8001"],
                 stdout=open("embed.log", "w"), stderr=subprocess.STDOUT)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
                 stderr=open("cf2.log", "w"))
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf2.log").read())
    if m:
        print("EMBED URL:", m.group(0) + "/v1")
        break

EMBED URL: https://americas-practitioner-blond-fluid.trycloudflare.com/v1
